In [1]:
import pandas as pd

df = pd.read_csv('../data/accepted_2007_to_2018Q4.csv', nrows=50)

df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# List of safe features to use for analysis
safe_features = ['loan_amnt',
                    'term',
                    'int_rate',
                    'installment',
                    'grade',
                    'emp_length',
                    'home_ownership',
                    'annual_inc',
                    'verification_status',
                    'loan_status',
                    'purpose',
                    'dti',
                    'delinq_2yrs',
                    'fico_range_low',
                    'fico_range_high',
                    'inq_last_6mths',
                    'open_acc',
                    'pub_rec',
                    'revol_bal',
                    'revol_util',
]

# Read the CSV file again, but this time only load the safe features
df_filtered = pd.read_csv('../data/accepted_2007_to_2018Q4.csv', usecols=safe_features)

print(f"Dataset shape after filtering: {df_filtered.shape}")

Dataset shape after filtering: (2260701, 20)


In [3]:
df_filtered['loan_status'].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [4]:
# Defining the default and paid statuses for loan_status
default_statuses = ['Charged Off', 'Default', 'Late (31-120 days)']
paid_statuses = ['Fully Paid']

# Create a new DataFrame that only includes rows with loan_status in the defined target statuses
all_target_statuses = default_statuses + paid_statuses
df_clean = df_filtered[df_filtered['loan_status'].isin(all_target_statuses)].copy()

# Create a new column 'is_default' that indicates whether the loan is in default (1) or not (0)
df_clean['is_default'] = df_clean['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)

# Drop the original 'loan_status' column as it is no longer needed
df_clean = df_clean.drop(columns=['loan_status'])

# Display the new row count and the breakdown of the target variable
print("New row count:", len(df_clean))
print("\nTarget breakdown:")
print(df_clean['is_default'].value_counts())
print("\nPercentage of Defaults:")
print(df_clean['is_default'].value_counts(normalize=True) * 100)

New row count: 1366817

Target breakdown:
is_default
0    1076751
1     290066
Name: count, dtype: int64

Percentage of Defaults:
is_default
0    78.777993
1    21.222007
Name: proportion, dtype: float64


In [5]:
# Check which columns have missing values across our 1.36M rows
missing_summary = df_clean.isnull().sum()

# Display columns that have at least one missing value along with their percentage
print("Columns with missing values:")
for col, count in missing_summary[missing_summary > 0].items():
    pct = (count / len(df_clean)) * 100
    print(f"{col}: {count} missing rows ({pct:.2f}%)")


Columns with missing values:
emp_length: 80367 missing rows (5.88%)
dti: 398 missing rows (0.03%)
inq_last_6mths: 1 missing rows (0.00%)
revol_util: 884 missing rows (0.06%)


In [6]:
# Fill missing employment length values with 'Unknown'
df_clean['emp_length'] = df_clean['emp_length'].fillna('Unknown')

# Drop rows with missing values in critical columns
df_clean = df_clean.dropna(subset=['inq_last_6mths', 'dti', 'revol_util'])

# After cleaning, check the number of remaining missing values and the number of rows left for modeling
remaining_missing = df_clean.isnull().sum().sum()
print(f"Total missing values left in dataset: {remaining_missing}")
print(f"Remaining rows for modeling: {len(df_clean)}")

Total missing values left in dataset: 0
Remaining rows for modeling: 1365535


In [7]:
# Display the data types of the cleaned DataFrame
df_clean.dtypes

loan_amnt              float64
term                    object
int_rate               float64
installment            float64
grade                   object
emp_length              object
home_ownership          object
annual_inc             float64
verification_status     object
purpose                 object
dti                    float64
delinq_2yrs            float64
fico_range_low         float64
fico_range_high        float64
inq_last_6mths         float64
open_acc               float64
pub_rec                float64
revol_bal              float64
revol_util             float64
is_default               int64
dtype: object

In [8]:
# Identify categorical columns for encoding
text_cols = df_clean.select_dtypes(include=['object']).columns

# Display the unique values and their counts for each categorical column
for col in text_cols:
    print(f"Column: {col}")
    print(df_clean[col].value_counts())
    print("\n")

Column: term
term
36 months    1032289
60 months     333246
Name: count, dtype: int64


Column: grade
grade
B    397261
C    388829
A    236303
D    205445
E     95700
F     32653
G      9344
Name: count, dtype: int64


Column: emp_length
emp_length
10+ years    448501
2 years      123643
< 1 year     109984
3 years      109334
1 year        89863
5 years       85427
4 years       81926
Unknown       79933
6 years       63610
8 years       61417
7 years       60353
9 years       51544
Name: count, dtype: int64


Column: home_ownership
home_ownership
MORTGAGE    674208
RENT        543467
OWN         147367
ANY            303
OTHER          142
NONE            48
Name: count, dtype: int64


Column: verification_status
verification_status
Source Verified    529991
Verified           424707
Not Verified       410837
Name: count, dtype: int64


Column: purpose
purpose
debt_consolidation    792209
credit_card           299140
home_improvement       88794
other                  79303
major_pu

In [9]:
# Cleaning the "term" column to convert it from a string to an integer
df_clean['term'] = df_clean['term'].str.extract('(\d+)').astype(int)

# Map 'grade' to numerical values for easier analysis
grade_mapping = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df_clean['grade'] = df_clean['grade'].map(grade_mapping)

# Map 'emp_length' to numerical values, treating 'Unknown' as -1
emp_length_mapping = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10,
    'Unknown': -1
}

df_clean['emp_length'] = df_clean['emp_length'].map(emp_length_mapping)

# One-hot encode the 'home_ownership', 'purpose' and 'verification_status' categorical columns.
# Changing df_clean to df_final to reflect that this is the final cleaned and encoded DataFrame ready for modeling.
df_final = pd.get_dummies(df_clean, columns=['home_ownership', 'purpose', 'verification_status'], drop_first=True, dtype=int)

print("Final DataFrame shape after encoding:", df_final.shape)
print("\nRemaining non-numeric columns after encoding:")
print(df_final.select_dtypes(include=['object']).columns.tolist())

Final DataFrame shape after encoding: (1365535, 37)

Remaining non-numeric columns after encoding:
[]


In [10]:
# X for the training features and y for the target variable
X = df_final.drop(columns=['is_default'])

y = df_final['is_default']

print(f"Feature matrix shape (x): {X.shape}")
print(f"Target vector shape (y): {y.shape}")

Feature matrix shape (x): (1365535, 36)
Target vector shape (y): (1365535,)


In [11]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets (80% train, 20% test)
# random_state = 42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data Split Summary:")
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Data Split Summary:
Training set size: 1092428
Testing set size: 273107


In [12]:
import xgboost as xgb

# Initialize the model

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    use_label_encoder=False,
)

model.fit(X_train, y_train)

print("Model training completed.")

C:\Users\Rhythm\AppData\Roaming\Python\Python310\site-packages\xgboost\training.py:200: UserWarning: [19:44:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model training completed.


In [13]:
from sklearn.metrics import classification_report, roc_auc_score

print("Evaluating model performance on the test set...")

# Make predictions on the test set
y_pred = model.predict(X_test)

# Get predicted probabilities for the positive class (default)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Print classification report and ROC AUC score

print("Credit Risk Model Performance on Test Set:")
print(classification_report(y_test, y_pred))

print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

Evaluating model performance on the test set...
Credit Risk Model Performance on Test Set:
              precision    recall  f1-score   support

           0       0.88      0.64      0.74    215154
           1       0.33      0.68      0.45     57953

    accuracy                           0.65    273107
   macro avg       0.61      0.66      0.59    273107
weighted avg       0.76      0.65      0.68    273107

ROC AUC Score: 0.7176


In [16]:
import shap
import numpy as np

X_sample = X_test.sample(1000, random_state=42)

explainer = shap.Explainer(model.predict, X_sample)
shap_values = explainer(X_sample)

# Generate SHAP summary plot for feature importance
feature_importance = np.abs(shap_values.values).mean(0)
importance_df = pd.DataFrame({
    'feature': X_test.columns, 
    'importance': feature_importance
    }).sort_values(by='importance', ascending=False)

print("\n Top 5 Drivers of Credit Risk (SHAP Feature Importance):")
print(importance_df.head(5).to_string(index=False))

PermutationExplainer explainer: 1001it [00:37, 22.24it/s]                         


 Top 5 Drivers of Credit Risk (SHAP Feature Importance):
  feature  importance
    grade    0.174617
     term    0.102562
 int_rate    0.097059
      dti    0.067314
loan_amnt    0.053278


In [23]:
test_probabilities = model.predict_proba(X_sample)[:, 1]
high_risk_index = np.argmax(test_probabilities)

customer_data = X_sample.iloc[high_risk_index]

raw_shap_array = shap_values[high_risk_index].values
if len(raw_shap_array.shape) > 1:
    raw_shap_array = raw_shap_array[:, 1] if raw_shap_array.shape[1] == 2 else raw_shap_array.flatten()

denial_reasons = pd.DataFrame({
    'feature': X_sample.columns,
    'Current Value': customer_data.values,
    'Risk Impact (SHAP Value)': raw_shap_array,
}).sort_values(by='Risk Impact (SHAP Value)', ascending=False)

print("=========================================")
print("🏦 REGULATORY ADVERSE ACTION NOTICE (CREDIT DENIAL)")
print("=========================================\n")
print(f"Applicant Sample ID: {X_sample.index[high_risk_index]}")
print(f"Model Calculated Default Probability: {test_probabilities[high_risk_index]:.2%}\n")
print("Dear Applicant,")
print("Thank you for your recent application. After careful review of your credit profile ")
print("via our underwriting systems, we regret to inform you that we cannot approve your ")
print("request at this time. The primary reasons contributing to this decision are:\n")

for feature_name, row in denial_reasons.head(2).iterrows():
    print(f" {str(feature_name).upper()} (Your Value: {row['Current Value']})")

print("\n=========================================")

🏦 REGULATORY ADVERSE ACTION NOTICE (CREDIT DENIAL)

Applicant Sample ID: 46515
Model Calculated Default Probability: 86.29%

Dear Applicant,
Thank you for your recent application. After careful review of your credit profile 
via our underwriting systems, we regret to inform you that we cannot approve your 
request at this time. The primary reasons contributing to this decision are:

 4 (Your Value: 6.0)
 1 (Your Value: 60.0)

